# IK end-effector control

This notebook replaces the terminal UI from `../../WBC/ik_pose_cli_v3.py` with a Jupyter widget panel. It sends small Cartesian end-effector increments through `ArmSdk.ik_move_EE()` and clamps joint changes per command with `max_dq`.


In [1]:
import os
import sys
from pathlib import Path

NOTEBOOK_DIR = Path.cwd().resolve()
MODULES_DIR = NOTEBOOK_DIR.parent
ROOT_DIR = MODULES_DIR.parent
for path in (str(MODULES_DIR), str(ROOT_DIR), str(MODULES_DIR / "scripts")):
    if path not in sys.path:
        sys.path.insert(0, path)

IFACE = os.environ.get("G1_IFACE", "eth0")
DOMAIN_ID = int(os.environ.get("G1_DOMAIN_ID", "0"))
print(f"Configured for iface={IFACE!r}, domain_id={DOMAIN_ID}.")


Configured for iface='eth0', domain_id=0.


Import `ArmSdk`, widgets, and numeric helpers.


In [2]:
import json
import numpy as np
import ipywidgets as widgets
from IPython.display import display

from arm_sdk import ArmSdk


Create the IK controller and sync it to the current measured upper-body state.


In [3]:
ik = ArmSdk(iface=IFACE, domain_id=DOMAIN_ID)
ik.resync()
print("ArmSdk IK controller synced to current state.")


ArmSdk IK controller synced to current state.


Helpers for EE pose display and single-increment commands.


In [4]:
DOF_INDEX = {"x": 0, "y": 1, "z": 2, "roll": 3, "pitch": 4, "yaw": 5}


def pose_summary(arm):
    T = ik.get_ee_pose(arm)
    return {
        "arm": arm,
        "position_xyz_m": [round(float(v), 4) for v in T[:3, 3]],
        "rotation_matrix": [[round(float(v), 4) for v in row] for row in T[:3, :3]],
    }


def apply_increment(arm, dof, signed_step, position_only=False, max_dq=0.12):
    inc = np.zeros(6, dtype=np.float64)
    inc[DOF_INDEX[dof]] = float(signed_step)
    selected_axis = DOF_INDEX[dof] if dof in {"x", "y", "z"} else None
    info = ik.ik_move_EE(
        inc,
        arm=arm,
        position_only=bool(position_only),
        selected_axis=selected_axis,
        max_dq=float(max_dq),
    )
    return info


Run the panel. Use small translation and rotation steps, and resync after physical contact or manual repositioning.


In [5]:
arm = widgets.ToggleButtons(options=["left", "right", "both"], value="right", description="Arm")
dof = widgets.Dropdown(options=list(DOF_INDEX), value="x", description="DOF")
translation_step = widgets.FloatSlider(value=0.02, min=0.002, max=0.08, step=0.002, description="m step")
rotation_step = widgets.FloatSlider(value=0.08, min=0.01, max=0.30, step=0.01, description="rad step")
max_dq = widgets.FloatSlider(value=0.12, min=0.02, max=0.25, step=0.01, description="max dq")
position_only = widgets.Checkbox(value=False, description="free orientation for xyz")
minus = widgets.Button(description="- Step")
plus = widgets.Button(description="+ Step", button_style="success")
resync = widgets.Button(description="Resync", button_style="info")
status = widgets.Textarea(layout=widgets.Layout(width="100%", height="260px"), disabled=True)


def current_step():
    return translation_step.value if dof.value in {"x", "y", "z"} else rotation_step.value


def refresh(extra=None):
    arms = ["left", "right"] if arm.value == "both" else [arm.value]
    payload = {"status": extra, "poses": [pose_summary(a) for a in arms]}
    status.value = json.dumps(payload, indent=2)


def move(sign):
    try:
        info = apply_increment(arm.value, dof.value, sign * current_step(), position_only.value, max_dq.value)
        refresh(info)
    except Exception as exc:
        refresh({"error": str(exc)})

minus.on_click(lambda _: move(-1.0))
plus.on_click(lambda _: move(+1.0))
resync.on_click(lambda _: (ik.resync(), refresh("resynced")))
refresh("ready")
display(widgets.VBox([widgets.HBox([arm, dof, position_only]), widgets.HBox([translation_step, rotation_step, max_dq]), widgets.HBox([minus, plus, resync]), status]))
